# Fine-tune NLLB-200-distilled-600M cho en→vi và ja→vi (Colab T4)

Notebook này fine-tune `facebook/nllb-200-distilled-600M` trên dữ liệu song ngữ hội thoại/phụ đề, có **ngữ cảnh câu trước**
(định dạng `"câu_trước câu_hiện_tại" → "câu_trước_vi câu_hiện_tại_vi"`, đúng như app dùng khi dịch), rồi xuất ra
CTranslate2 int8 để giải nén thẳng vào `models/mt/` của app.

**Thứ tự:** cấu hình → mount Drive + resume → tải dữ liệu → làm sạch (lọc độ dài, tag, ngôn ngữ, LaBSE) →
(tuỳ chọn) pivot ja-en→vi → ghép ngữ cảnh → huấn luyện `Seq2SeqTrainer` → đánh giá BLEU/chrF gốc vs fine-tune →
xuất CTranslate2 int8 → zip `nllb-vi-ct2-int8.zip` và tải về.

**Yêu cầu:** Runtime → Change runtime type → **T4 GPU**. Checkpoint, dữ liệu đã làm sạch và kết quả đánh giá đều lưu trên
Google Drive (`MyDrive/nllb_vi_finetune`), nên khi Colab ngắt kết nối chỉ cần **Run all** lại: các bước đã xong sẽ được bỏ qua và
huấn luyện chạy tiếp từ checkpoint cuối.

**License:** model gốc NLLB-200 là **CC-BY-NC-4.0 (phi thương mại)**, model fine-tune cũng chỉ dùng phi thương mại.
License từng bộ dữ liệu ghi trong ô cấu hình và trong `training/data_sources.md`.

## 1. Cài thư viện
Torch có sẵn trên Colab nên không cài lại. Nếu pip báo xung đột phiên bản, có thể bỏ số phiên bản (`==...`) rồi chạy lại.

In [ ]:
!pip -q install "transformers==5.21.0" "accelerate==1.15.0" "sentencepiece==0.2.2" "sacrebleu==2.6.0" "sacremoses==0.2.0" "ctranslate2==4.8.4" "sentence-transformers==6.2.0"

## 2. Cấu hình
Bật/tắt từng bộ dữ liệu bằng `True`/`False`. Smoke test nhanh: `MAX_SAMPLES = 2000`, `EPOCHS = 1`.

| Khoá | Cặp | License | Mặc định |
|---|---|---|---|
| `iwslt15_en_vi` | en-vi | TED Talks Usage Policy (CC BY-NC-ND 4.0) | bật |
| `opensubtitles_en_vi` | en-vi | OPUS OpenSubtitles v2018, không ghi license; dẫn nguồn opensubtitles.org | bật |
| `ted2020_en_vi` | en-vi | TED Talks Usage Policy (CC BY-NC-ND 4.0) | bật |
| `phomt_en_vi` | en-vi | **chỉ nghiên cứu** (điều khoản VinAI, cần token HF + chấp nhận điều khoản) | **tắt** |
| `opensubtitles_ja_vi` | ja-vi | như OpenSubtitles ở trên | bật |
| `ted2020_ja_vi` | ja-vi | TED Talks Usage Policy (CC BY-NC-ND 4.0) | bật |
| `alt_ja_vi` | ja-vi | CC BY 4.0 | bật |
| pivot ja-en→vi | ja-vi | OpenSubtitles en-ja + bản dịch máy của NLLB gốc | tắt |

In [ ]:
# CELL: config
PAIRS = ["en-vi", "ja-vi"]
DATASETS = {
    "iwslt15_en_vi": True,
    "opensubtitles_en_vi": True,
    "ted2020_en_vi": True,
    "phomt_en_vi": False,  # research only (VinAI terms); needs HF token + accepted terms
    "opensubtitles_ja_vi": True,
    "ted2020_ja_vi": True,
    "alt_ja_vi": True,
}
MAX_SAMPLES = 40_000          # max raw pairs read per dataset (2000 for a smoke test)
HELDOUT_PER_DATASET = 300     # test pairs held out from datasets without an official test split
TEST_FILM_CAP = 60            # OpenSubtitles: max test lines per film (300 lines => at least 5 films)
EVAL_MAX_PER_PAIR = 500       # test sentences per language pair for BLEU/chrF
SAMPLE_SENTENCES = 10         # side-by-side examples printed per pair

USE_LABSE = True              # LaBSE cosine filter (GPU, a few minutes)
LABSE_MIN_SIM = 0.7
USE_PIVOT_JA_EN = False       # synthesize ja-vi from OpenSubtitles ja-en with the base NLLB
PIVOT_MAX = 20_000

CONTEXT_N = 1                 # previous sentences joined as context (app default: 1)
CONTEXT_PROB = 0.5            # share of samples (with a previous sentence) that carry context

EPOCHS = 2
LR = 5e-5
BATCH = 8
GRAD_ACCUM = 4
FP16 = True
OPTIM = "adafactor"           # small optimizer state => ~2.5 GB per checkpoint ("adamw_torch" ~7 GB)
WARMUP_STEPS = 200
MAX_LEN = 128                 # tokens; longer samples are dropped (not truncated)
SAVE_STEPS = 2000             # local save interval; a save may be copied to ONE fixed Drive folder (replaced)
DRIVE_SYNC_MIN = 60           # copy a checkpoint to Drive at most once per this many minutes (first save always)
DRIVE_SYNC_MAX = 2            # max Drive copies per session (each replaced copy, ~2.5 GB, sits in Drive Trash)
SEED = 42

BASE_MODEL = "facebook/nllb-200-distilled-600M"
DRIVE_DIR = "/content/drive/MyDrive/nllb_vi_finetune"
EXPORT_NAME = "nllb-vi-ct2-int8"
LANG_CODES = {"en": "eng_Latn", "ja": "jpn_Jpan", "vi": "vie_Latn"}

## 3. Mount Google Drive + resume
Mỗi cấu hình dữ liệu/huấn luyện có một mã băm riêng; checkpoint và dữ liệu đã làm sạch lưu theo mã đó nên đổi cấu hình
sẽ không resume nhầm checkpoint cũ.

Checkpoint được ghi ra ổ local `/content/checkpoints` rồi chép lên **một** thư mục cố định trên Drive
(`ckpt_latest_<mã>`), bản mới thay bản cũ. Khi chạy lại, ô này chép checkpoint từ Drive về `/content` để huấn luyện tiếp.

**Lưu ý dung lượng Drive:** file bị xoá trên Drive chuyển vào **Thùng rác** và vẫn tính vào hạn mức 15 GB. Để giới hạn,
checkpoint chỉ được chép lên Drive tối đa một lần mỗi `DRIVE_SYNC_MIN` = 60 phút (lần lưu đầu tiên luôn chép), tối đa
`DRIVE_SYNC_MAX` = 2 lần mỗi phiên, và không chép ở bước cuối (model cuối được lưu ngay sau đó).

Ước tính dung lượng Drive tối đa với cấu hình mặc định, trong **một phiên** Colab:

| Thành phần | Dung lượng |
|---|---|
| Checkpoint trên Drive (trọng số fp32 615M tham số ≈ 2,46 GB + trạng thái Adafactor) | ≈ 2,5 GB |
| Bản checkpoint cũ trong Thùng rác (tối đa `DRIVE_SYNC_MAX` − 1 = 1 bản) | ≈ 2,5 GB |
| Model fine-tune cuối (`final_*`, fp16) | ≈ 1,2 GB |
| Bản zip CTranslate2 int8 chép lên Drive | ≈ 0,6 GB |
| Dữ liệu đã làm sạch + kết quả đánh giá | < 0,1 GB |
| **Tổng** | **≈ 6,9 GB** |

Mỗi lần chạy lại (phiên mới) có thể thêm tối đa 2 × 2,5 GB vào Thùng rác. Vì vậy trước khi chạy lại, hãy vào
https://drive.google.com/drive/trash và xoá vĩnh viễn các thư mục `ckpt_latest_*` cũ (hoặc "Dọn sạch thùng rác").

In [ ]:
# CELL: drive
import glob, hashlib, json, os, re, shutil

try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except ImportError:  # local dry-run
    IN_COLAB = False
    DRIVE_DIR = os.path.abspath(os.environ.get("NLLB_WORK_DIR", "nllb_work"))
os.makedirs(DRIVE_DIR, exist_ok=True)


def _hash(obj) -> str:
    return hashlib.sha1(json.dumps(obj, sort_keys=True).encode()).hexdigest()[:10]


DATA_CFG = dict(pairs=PAIRS, datasets=DATASETS, max_samples=MAX_SAMPLES, heldout=HELDOUT_PER_DATASET,
                eval_max=EVAL_MAX_PER_PAIR, labse=USE_LABSE, labse_min=LABSE_MIN_SIM,
                pivot=USE_PIVOT_JA_EN, pivot_max=PIVOT_MAX, ctx_n=CONTEXT_N, ctx_p=CONTEXT_PROB, film_cap=TEST_FILM_CAP, seed=SEED, v=3)
DATA_HASH = _hash(DATA_CFG)
RUN_HASH = _hash(dict(data=DATA_HASH, model=BASE_MODEL, epochs=EPOCHS, lr=LR, batch=BATCH, accum=GRAD_ACCUM,
                      fp16=FP16, optim=OPTIM, warmup=WARMUP_STEPS, max_len=MAX_LEN))
RAW_DIR = "/content/raw" if IN_COLAB else f"{DRIVE_DIR}/raw"   # raw downloads: local disk, re-downloadable
DATA_FILE = f"{DRIVE_DIR}/data/data_{DATA_HASH}.json"
LOCAL_CKPT = "/content/checkpoints" if IN_COLAB else f"{DRIVE_DIR}/local_ckpt"  # Trainer output_dir (local disk)
DRIVE_CKPT = f"{DRIVE_DIR}/ckpt_latest_{RUN_HASH}"   # the single checkpoint copy kept on Drive
FINAL_DIR = f"{DRIVE_DIR}/final_{RUN_HASH}"
DONE_FLAG = f"{FINAL_DIR}/DONE"                      # written last, after weights + tokenizer
BASE_EVAL_FILE = f"{DRIVE_DIR}/data/eval_base_{DATA_HASH}.json"


def _nonempty(f: str) -> bool:
    return os.path.isfile(f) and os.path.getsize(f) > 0


def ckpt_ok(p: str) -> bool:
    """A usable checkpoint has trainer state, non-empty weights and optimizer state."""
    return all(_nonempty(f"{p}/{f}") for f in ("trainer_state.json", "model.safetensors", "optimizer.pt"))


def ckpt_step(p: str) -> int:
    with open(f"{p}/trainer_state.json", encoding="utf-8") as f:
        return int(json.load(f)["global_step"])


def last_checkpoint(d: str):
    """Newest complete checkpoint-N in d; incomplete ones (cut off mid-save) are skipped."""
    cks = [p for p in glob.glob(f"{d}/checkpoint-*") if re.search(r"checkpoint-(\d+)$", p)]
    for p in sorted(cks, key=lambda p: int(p.rsplit("-", 1)[1]), reverse=True):
        if ckpt_ok(p):
            return p
        print(f"Bỏ qua checkpoint không đầy đủ: {p}")
    return None


def sync_to_drive(ckpt: str):
    """Copy a local checkpoint to DRIVE_CKPT: write a temp folder, then swap by rename (one copy on Drive)."""
    tmp, old = DRIVE_CKPT + ".tmp", DRIVE_CKPT + ".old"
    shutil.rmtree(tmp, ignore_errors=True)
    shutil.copytree(ckpt, tmp)
    shutil.rmtree(old, ignore_errors=True)
    if os.path.isdir(DRIVE_CKPT):
        os.rename(DRIVE_CKPT, old)
    os.rename(tmp, DRIVE_CKPT)
    shutil.rmtree(old, ignore_errors=True)
    print(f"Đã chép {os.path.basename(ckpt)} lên {DRIVE_CKPT}")


def restore_from_drive():
    """Copy the Drive checkpoint into LOCAL_CKPT unless a newer local one exists; return the local path."""
    local = last_checkpoint(LOCAL_CKPT)
    src = next((p for p in (DRIVE_CKPT, DRIVE_CKPT + ".old") if ckpt_ok(p)), None)  # .old: swap was interrupted
    if src and (local is None or ckpt_step(src) > ckpt_step(local)):
        dst = f"{LOCAL_CKPT}/checkpoint-{ckpt_step(src)}"
        shutil.rmtree(dst, ignore_errors=True)
        shutil.copytree(src, dst)
        print(f"Đã chép checkpoint từ Drive về {dst}")
        local = dst
    return local


DATA_READY = os.path.isfile(DATA_FILE)
TRAIN_DONE = os.path.isfile(DONE_FLAG)
RESUME_CKPT = None if TRAIN_DONE else restore_from_drive()
print(f"Thư mục làm việc: {DRIVE_DIR}")
print(f"Dữ liệu đã làm sạch: {'có, sẽ bỏ qua tải/làm sạch' if DATA_READY else 'chưa có'} ({DATA_FILE})")
print(f"Model fine-tune: {'đã xong' if TRAIN_DONE else 'chưa'}; checkpoint để resume: {RESUME_CKPT or 'không có'}")

## 4. Hàm làm sạch
Ô dưới ghi `cleaning.py` (bản sao y hệt `training/cleaning.py` trong repo, có test) để notebook chạy độc lập trên Colab.

In [ ]:
%%writefile cleaning.py
"""Parallel-corpus cleaning helpers for the NLLB fine-tune notebook (stdlib only, standalone)."""
from __future__ import annotations

import html
import re
import unicodedata
from collections.abc import Hashable, Iterable

Pair = tuple[str, str]

MAX_CHARS = 200
MAX_RATIO = 3.0
CJK_WEIGHT = 3  # one CJK/kana char carries roughly as much content as 3 Latin chars

_TAG = re.compile(r"<[^>]*>")
_ASS = re.compile(r"\{[^}]*\}")  # subtitle override tags such as {\an8}
_MUSIC = re.compile(r"[♪♫♬♩]")
_WS = re.compile(r"\s+")
_VI_CHARS = set(
    "ăâđêôơưàáảãạằắẳẵặầấẩẫậèéẻẽẹềếểễệìíỉĩịòóỏõọồốổỗộờớởỡợùúủũụừứửữựỳýỷỹỵ"
)


def _is_cjk(ch: str) -> bool:
    o = ord(ch)
    return (0x3040 <= o <= 0x30FF or 0x3400 <= o <= 0x4DBF or 0x4E00 <= o <= 0x9FFF
            or 0xF900 <= o <= 0xFAFF or 0xFF66 <= o <= 0xFF9F)  # kana + CJK ideographs; Hangul excluded


def _is_latin(ch: str) -> bool:
    o = ord(ch)
    return ch.isalpha() and (o < 0x250 or 0x1E00 <= o <= 0x1EFF)


def clean_text(s: str) -> str:
    """Unescape entities, drop HTML/subtitle tags and music symbols, normalise whitespace (NFC)."""
    s = html.unescape(html.unescape(s))
    s = _ASS.sub(" ", _TAG.sub(" ", s))
    s = _MUSIC.sub(" ", s)
    s = "".join(ch for ch in s if unicodedata.category(ch)[0] != "C" or ch in "\t\n")
    return _WS.sub(" ", unicodedata.normalize("NFC", s)).strip()


def weighted_len(s: str) -> int:
    """Character length where CJK/kana count CJK_WEIGHT (ja has no spaces and denser script)."""
    return sum(CJK_WEIGHT if _is_cjk(ch) else 1 for ch in s)


def clean_pair(src: str, tgt: str) -> Pair | None:
    """Clean one pair; None if empty, mojibake, longer than MAX_CHARS or length ratio outside [1/3, 3]."""
    if "�" in src + tgt or any(unicodedata.category(c) == "Cs" for c in src + tgt):
        return None
    src, tgt = clean_text(src), clean_text(tgt)
    if not src or not tgt or len(src) > MAX_CHARS or len(tgt) > MAX_CHARS:
        return None
    ratio = weighted_len(tgt) / weighted_len(src)
    if ratio > MAX_RATIO or ratio < 1 / MAX_RATIO:
        return None
    return src, tgt


def lang_ok(text: str, lang: str) -> bool:
    """Script-based language check for 'en', 'ja', 'vi' (no model download needed)."""
    letters = [ch for ch in text if ch.isalpha()]
    if not letters:
        return False
    cjk = sum(_is_cjk(ch) for ch in letters)
    latin = sum(_is_latin(ch) for ch in letters)
    if lang == "ja":
        return cjk / len(letters) >= 0.5
    if cjk or latin / len(letters) < 0.8:
        return False
    if lang == "vi" and len(text.split()) >= 4:
        return any(ch in _VI_CHARS for ch in text.lower())  # long vi lines virtually always carry diacritics
    return lang in ("en", "vi")


def make_context_pairs(rows: list[Pair], n: int = 1, sep: str = " ") -> list[Pair]:
    """For consecutive rows of ONE document, prepend up to n previous rows: "prev cur" -> "prev_vi cur_vi"."""
    out = []
    for i, (s, t) in enumerate(rows):
        ctx = rows[max(0, i - n):i] if n > 0 else []
        out.append((sep.join([c[0] for c in ctx] + [s]), sep.join([c[1] for c in ctx] + [t])))
    return out


def split_segments(items: Iterable[tuple[Hashable, Pair | None]]) -> list[list[Pair]]:
    """Group (doc_id, pair-or-None) into runs of consecutive kept pairs within the same document."""
    segs: list[list[Pair]] = []
    cur: list[Pair] = []
    last_doc: object = object()
    for doc, pair in items:
        if pair is None or doc != last_doc:
            if cur:
                segs.append(cur)
            cur = []
        if pair is not None:
            cur.append(pair)
        last_doc = doc
    if cur:
        segs.append(cur)
    return segs


def norm_key(text: str) -> str:
    """Leak-detection key: NFKC, lowercase, without punctuation, symbols and whitespace."""
    s = unicodedata.normalize("NFKC", text).lower()
    return "".join(ch for ch in s if not ch.isspace() and unicodedata.category(ch)[0] not in "PS")


def leak_keys(pairs: Iterable[Pair]) -> tuple[set[str], set[str]]:
    """(source keys, target keys) of the test pairs; empty keys are left out."""
    src, tgt = set(), set()
    for s, t in pairs:
        src.add(norm_key(s))
        tgt.add(norm_key(t))
    return src - {""}, tgt - {""}


def touches_test(rows: Iterable[Pair], keys: tuple[set[str], set[str]]) -> bool:
    """True if any row's source or target matches a test sentence (normalised)."""
    ks, kt = keys
    return any(norm_key(s) in ks or norm_key(t) in kt for s, t in rows)


def dedup_norm(pairs: Iterable[Pair]) -> list[Pair]:
    """Keep the first pair per normalised source (for test sets); pairs with an empty key are dropped."""
    seen: set[str] = set()
    out = []
    for s, t in pairs:
        k = norm_key(s)
        if k and k not in seen:
            seen.add(k)
            out.append((s, t))
    return out


def should_sync(step: int, max_steps: int, last_sync_ts: float | None, now: float, min_minutes: float,
                n_synced: int = 0, max_syncs: int | None = None) -> bool:
    """Whether a checkpoint saved at `step` is copied to Drive.

    Never at/after the last step (the final model is saved right after); the first save always syncs;
    later ones only after min_minutes since the previous sync and while fewer than max_syncs were made.
    """
    if max_steps and step >= max_steps:
        return False
    if max_syncs is not None and n_synced >= max_syncs:
        return False
    return last_sync_ts is None or now - last_sync_ts >= min_minutes * 60


def dedup_pairs(pairs: Iterable[Pair]) -> list[Pair]:
    """Drop exact duplicate pairs, keeping the first occurrence and order."""
    seen: set[Pair] = set()
    out = []
    for p in pairs:
        if p not in seen:
            seen.add(p)
            out.append(p)
    return out

In [ ]:
# CELL: import_cleaning
import importlib
import cleaning
importlib.reload(cleaning)
from cleaning import (clean_pair, dedup_norm, dedup_pairs, lang_ok, leak_keys, make_context_pairs, should_sync,
                      split_segments, touches_test)

## 5. Tokenizer và hàm dịch dùng chung
Mẫu được mã hoá giống hệt cách app tạo request cho CTranslate2: nguồn = `[mã_nguồn] + token + [</s>]`,
đích = `[vie_Latn] + token + [</s>]`.

In [ ]:
# CELL: tokenizer
import numpy as np
from transformers import AutoTokenizer
from transformers.utils import logging as hf_logging

hf_logging.set_verbosity_error()
tok = AutoTokenizer.from_pretrained(BASE_MODEL)
LANG_ID = {k: tok.convert_tokens_to_ids(v) for k, v in LANG_CODES.items()}
EOS_ID, PAD_ID = tok.eos_token_id, tok.pad_token_id
assert tok.unk_token_id not in LANG_ID.values(), LANG_ID


def tokenize_all(samples, chunk: int = 20_000):
    """[(src_lang, src, tgt)] -> [(src_ids, tgt_ids)] as int32 arrays; drops samples longer than MAX_LEN."""
    out = []
    for b in range(0, len(samples), chunk):
        part = samples[b:b + chunk]
        S = tok([s for _, s, _ in part], add_special_tokens=False)["input_ids"]
        T = tok([t for _, _, t in part], add_special_tokens=False)["input_ids"]
        for (lang, _, _), s, t in zip(part, S, T):
            if len(s) + 2 <= MAX_LEN and len(t) + 2 <= MAX_LEN:
                out.append((np.array([LANG_ID[lang]] + s + [EOS_ID], np.int32),
                            np.array([LANG_ID["vi"]] + t + [EOS_ID], np.int32)))
    print(f"Đã mã hoá {len(out)}/{len(samples)} mẫu (bỏ {len(samples) - len(out)} mẫu dài hơn {MAX_LEN} token)")
    return out


def device():
    import torch
    return "cuda" if torch.cuda.is_available() else "cpu"


def load_hf(path):
    import torch
    from transformers import AutoModelForSeq2SeqLM
    dtype = torch.float16 if device() == "cuda" else torch.float32
    return AutoModelForSeq2SeqLM.from_pretrained(path, dtype=dtype).to(device()).eval()


def free_gpu():
    """Call after `del`-ing model references."""
    import gc, torch
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def hf_translate(model, texts, src, tgt="vi", batch_size=32, beams=2, max_new=128):
    """Batched generation (length-sorted) with the same request layout as the app."""
    import torch
    order = sorted(range(len(texts)), key=lambda i: len(texts[i]))
    res = [""] * len(texts)
    for b in range(0, len(order), batch_size):
        idx = order[b:b + batch_size]
        ids = [[LANG_ID[src]] + tok.encode(texts[i], add_special_tokens=False)[:MAX_LEN - 2] + [EOS_ID] for i in idx]
        n = max(map(len, ids))
        inp = torch.tensor([x + [PAD_ID] * (n - len(x)) for x in ids], device=model.device)
        with torch.no_grad():
            gen = model.generate(input_ids=inp, attention_mask=(inp != PAD_ID).long(), num_beams=beams,
                                 forced_bos_token_id=LANG_ID[tgt], max_new_tokens=max_new)
        for i, d in zip(idx, tok.batch_decode(gen, skip_special_tokens=True)):
            res[i] = d.strip()
    return res

## 6. Tải dữ liệu
Nguồn (đã kiểm tra): OPUS (`object.pouta.csc.fi`), IWSLT'15 (bản Stanford, mirror PaddleNLP có kiểm MD5),
ALT (Hugging Face `mutiyama/alt`, nhánh parquet), PhoMT (Hugging Face `vinai/PhoMT`, gated).
Dữ liệu được đọc dạng luồng và dừng ở `MAX_SAMPLES` nên không tốn RAM.
Muốn bật PhoMT: chấp nhận điều khoản tại https://huggingface.co/datasets/vinai/PhoMT rồi thêm `HF_TOKEN` vào Colab Secrets (biểu tượng chìa khoá ở thanh bên trái).

In [ ]:
# CELL: download
import io, random, shutil, tarfile, time, zipfile
import urllib.request

OPUS = "https://object.pouta.csc.fi"
ALT_REV = "78f393e562d2940de035355229a7438192a06231"   # refs/convert/parquet of mutiyama/alt
IWSLT_URLS = ["https://paddlenlp.bj.bcebos.com/datasets/iwslt15.en-vi.tar.gz"]
IWSLT_MD5 = "aca22dc3f90962e42916dbb36d8f3e8e"
IWSLT_STANFORD = "https://nlp.stanford.edu/projects/nmt/data/iwslt15.en-vi"


def _md5(path):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()


def fetch(urls, name, md5=None):
    """Download the first working URL into RAW_DIR/name (cached, atomic, retried)."""
    path = os.path.join(RAW_DIR, name)
    if os.path.isfile(path):
        return path
    os.makedirs(RAW_DIR, exist_ok=True)
    for url in urls:
        for attempt in range(3):
            try:
                tmp = path + ".part"
                with urllib.request.urlopen(url, timeout=60) as r, open(tmp, "wb") as f:
                    shutil.copyfileobj(r, f, 1 << 20)
                if md5 and _md5(tmp) != md5:
                    raise IOError("sai MD5")
                os.replace(tmp, path)
                return path
            except Exception as e:
                print(f"  lỗi tải {url} (lần {attempt + 1}): {e}")
                time.sleep(3)
    raise RuntimeError(f"Không tải được {name}")


def split_tail(rows, n_test, window=2000):
    """Hold out the last n_test rows as test, cutting at a document boundary when one is near."""
    if n_test <= 0 or len(rows) <= 2 * n_test:
        return rows, []
    cut = len(rows) - n_test
    for j in range(cut, max(cut - window, 1), -1):
        if rows[j][0] != rows[j - 1][0]:
            cut = j
            break
    return rows[:cut], rows[cut:]


def _ids_doc(line):
    """OpenSubtitles .ids line -> (film, doc): film = 'en/YEAR/ID' dir of field 0, doc = film + file pair."""
    f = line.split("\t")
    return os.path.dirname(f[0]), "\t".join([os.path.dirname(f[0])] + f[:2])


def opus_rows(corpus, version, src, tgt, limit, n_test=0, seed=SEED, film_cap=TEST_FILM_CAP):
    """OPUS moses zip -> (train_rows, test_rows) of (doc_id, src_text, tgt_text).

    With a .ids file (OpenSubtitles) whole films are sampled at random across the WHOLE stream (pass 1 reads only
    the ids to count lines per film; pass 2 streams just the chosen films), and test films are disjoint from train
    films; each test film gives at most film_cap lines so the test set spans several films. Without ids the first limit + n_test lines are read and the tail is held out.
    """
    pair = "-".join(sorted((src, tgt)))  # OPUS orders codes alphabetically
    path = fetch([f"{OPUS}/OPUS-{corpus}/{version}/moses/{pair}.txt.zip"], f"{corpus}.{pair}.txt.zip")
    # newline="\n": a stray \r inside a line must not desynchronise the two sides
    text = lambda z, name: io.TextIOWrapper(z.open(name), "utf-8", errors="replace", newline="\n")
    with zipfile.ZipFile(path) as z:
        ids_name = f"{corpus}.{pair}.ids"
        has_ids = ids_name in z.namelist()
        role, stop = {}, None
        if has_ids:
            count, last_line = {}, {}
            for i, line in enumerate(text(z, ids_name)):
                film = _ids_doc(line)[0]
                count[film] = count.get(film, 0) + 1
                last_line[film] = i
            films = list(count)
            random.Random(seed).shuffle(films)
            n_tr = n_te = 0
            for film in films:
                if n_tr < limit:
                    role[film], n_tr = "train", n_tr + count[film]
                elif n_te < n_test:
                    role[film], n_te = "test", n_te + min(count[film], film_cap)
                else:
                    break
            stop = max((last_line[f] for f in role), default=-1)
            print(f"  {corpus} {pair}: chọn {len(role)}/{len(films)} phim")
        train, test, per_film = [], [], {}
        fi = text(z, ids_name) if has_ids else None
        for i, (ls, lt) in enumerate(zip(text(z, f"{corpus}.{pair}.{src}"), text(z, f"{corpus}.{pair}.{tgt}"))):
            if has_ids:
                if i > stop:
                    break
                film, doc = _ids_doc(fi.readline())
                r = role.get(film)
                if r == "train":
                    train.append((doc, ls.rstrip("\n"), lt.rstrip("\n")))
                elif r == "test" and per_film.get(film, 0) < film_cap:
                    per_film[film] = per_film.get(film, 0) + 1
                    test.append((doc, ls.rstrip("\n"), lt.rstrip("\n")))
            else:
                if len(train) >= limit + n_test:
                    break
                train.append((corpus, ls.rstrip("\n"), lt.rstrip("\n")))
    return (train, test) if has_ids else split_tail(train, n_test)


def iwslt_rows(limit):
    """IWSLT'15 en-vi (Stanford preprocessing: Moses-tokenized) -> detokenized train rows + tst2013 test rows."""
    from sacremoses import MosesDetokenizer
    names = ["train.en", "train.vi", "tst2013.en", "tst2013.vi"]
    try:
        with tarfile.open(fetch(IWSLT_URLS, "iwslt15.en-vi.tar.gz", md5=IWSLT_MD5)) as t:
            files = {n: t.extractfile(f"iwslt15.en-vi/{n}").read().decode("utf-8").split("\n") for n in names}
    except Exception as e:
        print(f"  mirror lỗi ({e}); thử Stanford")
        files = {n: open(fetch([f"{IWSLT_STANFORD}/{n}"], f"iwslt15.{n}"), encoding="utf-8").read().split("\n")
                 for n in names}
    det = {"en": MosesDetokenizer("en"), "vi": MosesDetokenizer("vi")}
    d = lambda s, lang: det[lang].detokenize(s.split())

    def rows(split, n):
        pairs = [(a, b) for a, b in zip(files[f"{split}.en"], files[f"{split}.vi"]) if a.strip() and b.strip()][:n]
        return [("iwslt", d(a, "en"), d(b, "vi")) for a, b in pairs]
    return rows("train", limit), rows("tst2013", 10**9)


def alt_rows(limit):
    """ALT ja-vi from the HF parquet conversion (official train/test split; doc id = SNT.URLID)."""
    import pandas as pd
    from huggingface_hub import hf_hub_download

    def load(split, n):
        p = hf_hub_download("mutiyama/alt", f"alt-parallel/{split}/0000.parquet", repo_type="dataset", revision=ALT_REV)
        df = pd.read_parquet(p, columns=["SNT.URLID", "translation"])
        out = [(doc, tr.get("ja"), tr.get("vi")) for doc, tr in zip(df["SNT.URLID"], df["translation"])]
        return [r for r in out if r[1] and r[2]][:n]
    return load("train", limit), load("test", 10**9)


def phomt_rows(limit):
    """PhoMT en-vi (gated, research only). Needs HF_TOKEN with the dataset terms accepted."""
    from huggingface_hub import hf_hub_download
    token = os.environ.get("HF_TOKEN")
    if IN_COLAB and not token:
        try:
            from google.colab import userdata
            token = userdata.get("HF_TOKEN")
        except Exception:
            token = None
    p = hf_hub_download("vinai/PhoMT", "PhoMT.zip", repo_type="dataset", token=token)
    with zipfile.ZipFile(p) as z:
        names = z.namelist()

        def find(split, lang):
            c = [n for n in names if n.endswith(f"{split}.{lang}")]
            c = [n for n in c if "detok" in n.lower()] or c
            if not c:
                raise FileNotFoundError(f"PhoMT.zip không có *{split}.{lang}")
            return c[0]

        def read(split, n):
            fa = io.TextIOWrapper(z.open(find(split, "en")), "utf-8", errors="replace", newline="\n")
            fb = io.TextIOWrapper(z.open(find(split, "vi")), "utf-8", errors="replace", newline="\n")
            out = []
            for a, b in zip(fa, fb):
                if len(out) >= n:
                    break
                out.append(("phomt", a.rstrip("\n"), b.rstrip("\n")))
            return out
        return read("train", limit), read("test", 2000)


H = HELDOUT_PER_DATASET
LOADERS = {
    "iwslt15_en_vi": ("en", lambda: iwslt_rows(MAX_SAMPLES)),
    "opensubtitles_en_vi": ("en", lambda: opus_rows("OpenSubtitles", "v2018", "en", "vi", MAX_SAMPLES, H)),
    "ted2020_en_vi": ("en", lambda: opus_rows("TED2020", "v1", "en", "vi", MAX_SAMPLES, H)),
    "phomt_en_vi": ("en", lambda: phomt_rows(MAX_SAMPLES)),
    "opensubtitles_ja_vi": ("ja", lambda: opus_rows("OpenSubtitles", "v2018", "ja", "vi", MAX_SAMPLES, H)),
    "ted2020_ja_vi": ("ja", lambda: opus_rows("TED2020", "v1", "ja", "vi", MAX_SAMPLES, H)),
    "alt_ja_vi": ("ja", lambda: alt_rows(MAX_SAMPLES)),
}

RAW = {}  # name -> (src_lang, train_rows, test_rows)
if DATA_READY:
    print("Đã có dữ liệu đã làm sạch trên Drive, bỏ qua bước tải.")
else:
    for name, on in DATASETS.items():
        lang, fn = LOADERS[name]
        if not on or f"{lang}-vi" not in PAIRS:
            continue
        t0 = time.time()
        try:
            tr, te = fn()
        except Exception as e:
            print(f"[{name}] BỎ QUA do lỗi: {type(e).__name__}: {e}")
            continue
        RAW[name] = (lang, tr, te)
        print(f"[{name}] train {len(tr)} / test {len(te)} cặp ({time.time() - t0:.0f}s)")

## 7. Làm sạch + lọc ngôn ngữ + lọc LaBSE
- `clean_pair`: bỏ tag HTML, `♪`, `{\an8}`; loại cặp rỗng, quá 200 ký tự, tỉ lệ độ dài > 3 hoặc < 1/3 (tiếng Nhật tính trọng số theo ký tự).
- `lang_ok`: kiểm tra chữ viết (Nhật phải có kana/kanji; câu tiếng Việt ≥ 4 từ phải có dấu, loại câu chưa dịch).
- LaBSE: giữ cặp có cosine ≥ `LABSE_MIN_SIM` (chạy theo batch trên GPU, tắt bằng `USE_LABSE = False`).

In [ ]:
# CELL: clean
def clean_rows(rows, src_lang):
    """[(doc, src, tgt)] -> [(doc, pair | None)]; None keeps the slot so context never skips a dropped line."""
    out = []
    for doc, s, t in rows:
        p = clean_pair(s, t)
        if p and not (lang_ok(p[0], src_lang) and lang_ok(p[1], "vi")):
            p = None
        out.append((doc, p))
    return out


def labse_filter(groups, min_sim=LABSE_MIN_SIM, batch_size=256):
    """groups: {name: [(doc, pair|None)]}; drops (sets None) pairs whose LaBSE cosine < min_sim, in place."""
    import torch
    from sentence_transformers import SentenceTransformer
    m = SentenceTransformer("sentence-transformers/LaBSE", device=device())
    if device() == "cuda":
        m.half()
    for name, items in groups.items():
        idx = [i for i, (_, p) in enumerate(items) if p]
        if not idx:
            continue
        enc = lambda xs: m.encode(xs, batch_size=batch_size, convert_to_tensor=True, normalize_embeddings=True,
                                  show_progress_bar=False)
        sims = []
        for b in range(0, len(idx), 20_000):
            part = idx[b:b + 20_000]
            es = enc([items[i][1][0] for i in part])
            et = enc([items[i][1][1] for i in part])
            sims += (es * et).sum(-1).float().cpu().tolist()
        dropped = 0
        for i, s in zip(idx, sims):
            if s < min_sim:
                items[i] = (items[i][0], None)
                dropped += 1
        print(f"[{name}] LaBSE loại {dropped}/{len(idx)} cặp")
    del m
    free_gpu()


if not DATA_READY:
    SRC_LANG = {name: lang for name, (lang, _, _) in RAW.items()}
    CLEAN = {name: clean_rows(tr, lang) for name, (lang, tr, _) in RAW.items()}
    TEST = {name: [p for _, p in clean_rows(te, lang) if p] for name, (lang, _, te) in RAW.items()}
    for name, items in CLEAN.items():
        kept = sum(p is not None for _, p in items)
        print(f"[{name}] giữ {kept}/{len(items)} cặp sau làm sạch + lọc ngôn ngữ; test {len(TEST[name])}")
    if USE_LABSE and CLEAN:
        labse_filter(CLEAN)

## 8. (Tuỳ chọn) Pivot ja-en → vi
Lấy OpenSubtitles ja-en, dịch phía tiếng Anh sang tiếng Việt bằng chính NLLB gốc để có thêm cặp ja-vi.
Bật bằng `USE_PIVOT_JA_EN = True` (khoảng 10–20 phút cho 20.000 câu trên T4).

In [ ]:
# CELL: pivot
if not DATA_READY and USE_PIVOT_JA_EN and "ja-vi" in PAIRS:
    rows, _ = opus_rows("OpenSubtitles", "v2018", "ja", "en", PIVOT_MAX, 0, seed=SEED + 1)
    kept = []
    for doc, ja, en in rows:
        p = clean_pair(ja, en)
        kept.append((doc, p[0], p[1]) if p and lang_ok(p[0], "ja") and lang_ok(p[1], "en") else (doc, "", ""))
    base = load_hf(BASE_MODEL)
    todo = [i for i, r in enumerate(kept) if r[2]]
    vi = hf_translate(base, [kept[i][2] for i in todo], "en", "vi", batch_size=64, beams=2)
    del base
    free_gpu()
    out = [(doc, "", "") for doc, _, _ in kept]
    for i, v in zip(todo, vi):
        out[i] = (kept[i][0], kept[i][1], v)
    CLEAN["pivot_ja_en"] = clean_rows(out, "ja")
    SRC_LANG["pivot_ja_en"] = "ja"
    if USE_LABSE:
        labse_filter({"pivot_ja_en": CLEAN["pivot_ja_en"]})
    print(f"[pivot_ja_en] {sum(p is not None for _, p in CLEAN['pivot_ja_en'])} cặp ja-vi tổng hợp")
else:
    print("Bỏ qua pivot.")

## 9. Ghép ngữ cảnh câu trước + lưu dữ liệu lên Drive
`make_context_pairs` ghép các câu liên tiếp trong cùng tài liệu: `"prev cur" → "prev_vi cur_vi"` (nối bằng dấu cách, kể cả
tiếng Nhật, giống cách app nối token ngữ cảnh). Khoảng `CONTEXT_PROB` số mẫu có ngữ cảnh, phần còn lại là câu đơn.

In [ ]:
# CELL: context
if not DATA_READY:
    rng = random.Random(SEED)
    test = {}
    for name, pairs in TEST.items():
        test.setdefault(SRC_LANG[name], []).extend([s, t, name] for s, t in pairs)
    for lang in test:
        keep = set(dedup_norm((r[0], r[1]) for r in test[lang]))  # first pair per normalised source
        test[lang] = [r for r in test[lang] if (r[0], r[1]) in keep and not keep.discard((r[0], r[1]))]
        rng.shuffle(test[lang])
        test[lang] = test[lang][:EVAL_MAX_PER_PAIR]
    # Leak guard on normalised text (case/punctuation/space-insensitive), source AND target side.
    keys = leak_keys((r[0], r[1]) for rows in test.values() for r in rows)
    train, stats = [], {}
    for name, items in CLEAN.items():
        lang, seen, n_ctx, n_leak = SRC_LANG[name], set(), 0, 0
        for seg in split_segments(items):
            ctx = make_context_pairs(seg, CONTEXT_N)
            for i, (plain, withctx) in enumerate(zip(seg, ctx)):
                if touches_test([plain], keys):
                    n_leak += 1
                    continue  # a test sentence never appears as the current sentence
                use_ctx = i > 0 and rng.random() < CONTEXT_PROB
                if use_ctx and touches_test(seg[max(0, i - CONTEXT_N):i], keys):
                    use_ctx = False  # nor as context: keep the plain sentence instead
                pair = withctx if use_ctx else plain
                if pair in seen:
                    continue
                seen.add(pair)
                n_ctx += use_ctx
                train.append((lang, pair[0], pair[1]))
        stats[name] = {"samples": len(seen), "with_context": n_ctx, "leak_dropped": n_leak}
        print(f"[{name}] {len(seen)} mẫu ({n_ctx} có ngữ cảnh; bỏ {n_leak} câu trùng tập test)")
    rng.shuffle(train)
    DATA = {"train": train, "test": test, "stats": stats, "config": DATA_CFG}
    os.makedirs(os.path.dirname(DATA_FILE), exist_ok=True)
    with open(DATA_FILE + ".part", "w", encoding="utf-8") as f:
        json.dump(DATA, f, ensure_ascii=False)
    os.replace(DATA_FILE + ".part", DATA_FILE)
    DATA_READY = True
else:
    with open(DATA_FILE, encoding="utf-8") as f:
        DATA = json.load(f)
print(f"Train: {len(DATA['train'])} mẫu; test: " + ", ".join(f"{k}-vi {len(v)}" for k, v in DATA["test"].items()))
for lang, s, t in DATA["train"][:3]:
    print(f"  [{lang}] {s}  →  {t}")

## 10. Huấn luyện (`Seq2SeqTrainer`)
fp16, batch 8 × gradient accumulation 4, lr 5e-5, 2 epoch. Mỗi `SAVE_STEPS` bước, checkpoint được lưu ra `/content` rồi
chép đè lên một thư mục duy nhất trên Drive; chạy lại notebook sau khi mất kết nối sẽ tiếp tục từ checkpoint đó.
Model cuối được coi là xong chỉ khi có file `DONE` (ghi sau cùng).

In [ ]:
# CELL: train
import shutil
import torch
from huggingface_hub import snapshot_download
from transformers import (AutoModelForSeq2SeqLM, Seq2SeqTrainer, Seq2SeqTrainingArguments, TrainerCallback,
                          set_seed)

TOK_FILES = ["sentencepiece.bpe.model", "tokenizer.json", "tokenizer_config.json", "special_tokens_map.json"]


class PairDataset(torch.utils.data.Dataset):
    def __init__(self, items):
        self.items = items

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        s, t = self.items[i]
        return {"input_ids": s, "labels": t}


def collate(batch):
    n = max(len(b["input_ids"]) for b in batch)
    m = max(len(b["labels"]) for b in batch)
    ids = torch.full((len(batch), n), PAD_ID, dtype=torch.long)
    att = torch.zeros((len(batch), n), dtype=torch.long)
    lab = torch.full((len(batch), m), -100, dtype=torch.long)
    for i, b in enumerate(batch):
        ids[i, :len(b["input_ids"])] = torch.from_numpy(b["input_ids"].astype(np.int64))
        att[i, :len(b["input_ids"])] = 1
        lab[i, :len(b["labels"])] = torch.from_numpy(b["labels"].astype(np.int64))
    return {"input_ids": ids, "attention_mask": att, "labels": lab}  # decoder inputs = labels shifted right


class DriveSync(TrainerCallback):
    """After a local save, replace the single Drive copy with it (rate-limited, never at the final step)."""

    def __init__(self):
        self.last_ts, self.n = None, 0

    def on_save(self, args, state, control, **kwargs):
        ckpt = f"{args.output_dir}/checkpoint-{state.global_step}"
        if not (state.is_world_process_zero and ckpt_ok(ckpt)):
            return
        now = time.time()
        if should_sync(state.global_step, state.max_steps, self.last_ts, now, DRIVE_SYNC_MIN, self.n, DRIVE_SYNC_MAX):
            sync_to_drive(ckpt)
            self.last_ts, self.n = now, self.n + 1
        else:
            print(f"Checkpoint {state.global_step} chỉ lưu local (chưa đến lượt chép lên Drive)")


if TRAIN_DONE:
    print(f"Model fine-tune đã có ở {FINAL_DIR}, bỏ qua huấn luyện.")
else:
    set_seed(SEED)
    train_items = tokenize_all(DATA["train"])
    model = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL)  # fp32 master weights; Trainer autocasts to fp16
    args = Seq2SeqTrainingArguments(
        output_dir=LOCAL_CKPT, num_train_epochs=EPOCHS, learning_rate=LR,
        per_device_train_batch_size=BATCH, gradient_accumulation_steps=GRAD_ACCUM,
        fp16=FP16 and torch.cuda.is_available(), optim=OPTIM, warmup_steps=WARMUP_STEPS,
        lr_scheduler_type="linear", logging_steps=50, save_strategy="steps", save_steps=SAVE_STEPS,
        save_total_limit=1, report_to="none", remove_unused_columns=False, dataloader_num_workers=2, seed=SEED)
    trainer = Seq2SeqTrainer(model=model, args=args, train_dataset=PairDataset(train_items), data_collator=collate,
                             callbacks=[DriveSync()])
    ckpt = restore_from_drive()
    print(f"Bắt đầu huấn luyện {'tiếp từ ' + ckpt if ckpt else 'từ đầu'}")
    trainer.train(resume_from_checkpoint=ckpt)
    # Tokenizer first (unchanged by fine-tuning: the original files, same as the app's models/mt), weights next,
    # DONE sentinel last, so an interrupted save is never taken for a finished model.
    os.makedirs(FINAL_DIR, exist_ok=True)
    tok_src = snapshot_download(BASE_MODEL, allow_patterns=TOK_FILES)
    for f in TOK_FILES:
        if os.path.isfile(os.path.join(tok_src, f)):
            shutil.copy(os.path.join(tok_src, f), FINAL_DIR)
    trainer.model.half().save_pretrained(FINAL_DIR)  # fp16 (~1.2 GB): enough for eval and the int8 export
    with open(DONE_FLAG, "w") as f:
        f.write(f"step {trainer.state.global_step}")
    TRAIN_DONE = True
    del trainer, model
    free_gpu()
    print(f"Đã lưu model fine-tune vào {FINAL_DIR}")

## 11. Đánh giá: BLEU/chrF (sacrebleu) model gốc vs fine-tune
Đánh giá trên tập test giữ riêng (IWSLT tst2013, ALT test, phần cuối OpenSubtitles/TED không dùng để train).
Kết quả lưu trên Drive nên chạy lại không phải dịch lại.

In [ ]:
# CELL: eval
import sacrebleu


def evaluate(model_path, cache_file):
    if os.path.isfile(cache_file):
        with open(cache_file, encoding="utf-8") as f:
            return json.load(f)
    model = load_hf(model_path)
    res = {}
    for lang, rows in DATA["test"].items():
        hyps = hf_translate(model, [r[0] for r in rows], lang)
        res[lang] = hyps
    del model
    free_gpu()
    with open(cache_file, "w", encoding="utf-8") as f:
        json.dump(res, f, ensure_ascii=False)
    return res


def scores(hyps, refs):
    return sacrebleu.corpus_bleu(hyps, [refs]).score, sacrebleu.corpus_chrf(hyps, [refs]).score


HYP = {"gốc": evaluate(BASE_MODEL, BASE_EVAL_FILE), "fine-tune": evaluate(FINAL_DIR, f"{FINAL_DIR}/eval_test.json")}
print(f"{'Cặp / bộ test':<28}{'n':>6}{'BLEU gốc':>10}{'BLEU FT':>9}{'chrF gốc':>10}{'chrF FT':>9}")
for lang, rows in DATA["test"].items():
    groups = {f"{lang}-vi (tất cả)": list(range(len(rows)))}
    for i, r in enumerate(rows):
        groups.setdefault(f"  {r[2]}", []).append(i)
    for g, idx in groups.items():
        refs = [rows[i][1] for i in idx]
        b0, c0 = scores([HYP["gốc"][lang][i] for i in idx], refs)
        b1, c1 = scores([HYP["fine-tune"][lang][i] for i in idx], refs)
        print(f"{g:<28}{len(idx):>6}{b0:>10.2f}{b1:>9.2f}{c0:>10.2f}{c1:>9.2f}")

for lang, rows in DATA["test"].items():
    print(f"\n===== {SAMPLE_SENTENCES} câu mẫu {lang}→vi =====")
    for i, r in enumerate(rows[:SAMPLE_SENTENCES]):
        print(f"[{i + 1}] Nguồn    : {r[0]}\n    Tham chiếu: {r[1]}\n    Gốc      : {HYP['gốc'][lang][i]}\n"
              f"    Fine-tune: {HYP['fine-tune'][lang][i]}")

## 12. Xuất CTranslate2 int8 + kiểm tra nhanh
Cấu trúc thư mục giống `models/mt/` mà `scripts/download_models.py` tải về: `model.bin`, `config.json`, vocab dùng chung,
`sentencepiece.bpe.model`, `tokenizer.json`, `tokenizer_config.json`, `special_tokens_map.json`.
Ô này cũng dịch thử bằng CTranslate2 theo đúng cách app gửi request (có ngữ cảnh câu trước).

In [ ]:
# CELL: export
import ctranslate2
from ctranslate2.converters import TransformersConverter

OUT_DIR = f"/content/{EXPORT_NAME}" if IN_COLAB else f"{DRIVE_DIR}/{EXPORT_NAME}"
copy = [f for f in TOK_FILES if os.path.isfile(os.path.join(FINAL_DIR, f))]
# FINAL_DIR holds fp16 weights: load them as fp16 explicitly, then quantize to int8.
TransformersConverter(FINAL_DIR, copy_files=copy, load_as_float16=True).convert(OUT_DIR, quantization="int8",
                                                                                force=True)
print("Các file:", sorted(os.listdir(OUT_DIR)))
assert os.path.isfile(f"{OUT_DIR}/model.bin") and os.path.isfile(f"{OUT_DIR}/sentencepiece.bpe.model")

try:
    ct2 = ctranslate2.Translator(OUT_DIR, device="cuda", compute_type="int8")
except Exception as e:
    print(f"CUDA lỗi ({e}), dùng CPU")
    ct2 = ctranslate2.Translator(OUT_DIR, device="cpu", compute_type="int8")
ct2_tok = AutoTokenizer.from_pretrained(OUT_DIR)


def ct2_translate(text, src, ctx=None):
    """Mirror of backend/mt/nllb.py: source=[src]+ctx_src+tokens+[</s>], target_prefix=[vie_Latn]+ctx_tgt."""
    toks = ct2_tok.convert_ids_to_tokens(ct2_tok.encode(text, add_special_tokens=False))
    cs, ctgt = ctx or ([], [])
    prefix = [LANG_CODES["vi"]] + ctgt
    res = ct2.translate_batch([[LANG_CODES[src]] + cs + toks + ["</s>"]], target_prefix=[prefix],
                              beam_size=2, max_decoding_length=256)
    hyp = [t for t in res[0].hypotheses[0][len(prefix):] if t != "</s>"]
    return toks, hyp, ct2_tok.convert_tokens_to_string(hyp).strip()


for src, lines in {"en": ["Hello, how are you?", "I'm going to the market today."],
                   "ja": ["こんにちは、元気ですか?", "今日は市場に行きます。"]}.items():
    ctx = None
    for line in lines:
        toks, hyp, out = ct2_translate(line, src, ctx)
        print(f"[{src}] {line}  →  {out}")
        ctx = (toks, hyp)

## 13. Nén zip và tải về
Trên máy Windows: **sao lưu** thư mục `models/mt/` hiện có, **xoá sạch** nội dung `models/mt/` (bản xuất dùng
`shared_vocabulary.json`, nếu để lại `shared_vocabulary.txt` cũ sẽ lẫn hai vocab), rồi giải nén `nllb-vi-ct2-int8.zip`
vào `models/mt/` sao cho `model.bin` nằm ngay trong `models/mt/`.
Bản zip cũng được chép lên Drive phòng khi tải trực tiếp bị lỗi.

In [ ]:
# CELL: zip
zip_path = shutil.make_archive(OUT_DIR, "zip", root_dir=OUT_DIR)  # files at the zip root
print(f"{zip_path}: {os.path.getsize(zip_path) / 2**20:.0f} MB")
if IN_COLAB:
    shutil.copy(zip_path, DRIVE_DIR)
    from google.colab import files
    files.download(zip_path)